# Music Generation with GANs

## 1.1 Dependencies

In [ ]:
# Import Tensorflow 2.0
#%tensorflow_version 2.x
!pip install --upgrade pip
!pip install --upgrade tensorflow
import tensorflow as tf

# Download and import the MIT 6.S191 package
!pip install mitdeeplearning
import mitdeeplearning as mdl

# Import all remaining packages
import numpy as np
import os
import time
import functools
from IPython import display as ipythondisplay
from tqdm import tqdm
!apt-get install abcmidi timidity > /dev/null 2>&1

## 1.2 Dataset

In [ ]:
# Download the dataset
songs = mdl.lab1.load_training_data()

# Print one of the songs to inspect it in greater detail!
example_song = songs[0]
print("\nExample song: ")
print(example_song)

In [ ]:
# Convert the ABC notation to audio file and listen to it
mdl.lab1.play_song(example_song)

In [ ]:
# Join our list of song strings into a single string containing all songs
songs_joined = "\n\n".join(songs)

# Find all unique characters in the joined string
vocab = sorted(set(songs_joined))
print("There are", len(vocab), "unique characters in the dataset")

## 1.3 Process the dataset for the learning task

### Vectorize the text

In [ ]:
### Define numerical representation of text ###

# Create a mapping from character to unique index.
# For example, to get the index of the character "d",
#   we can evaluate `char2idx["d"]`.
char2idx = {u:i for i, u in enumerate(vocab)}

# Create a mapping from indices to characters. This is
#   the inverse of char2idx and allows us to convert back
#   from unique index to the character in our vocabulary.
idx2char = np.array(vocab)

In [ ]:
print('{')
for char,_ in zip(char2idx, range(20)):
    print('  {:4s}: {:3d},'.format(repr(char), char2idx[char]))
print('  ...\n}')

In [ ]:
### Vectorize the songs string ###

'''TODO: Write a function to convert the all songs string to a vectorized
    (i.e., numeric) representation. Use the appropriate mapping
    above to convert from vocab characters to the corresponding indices.

  NOTE: the output of the `vectorize_string` function
  should be a np.array with `N` elements, where `N` is
  the number of characters in the input string
'''

def vectorize_string(string):
  vectorized_songs = np.array([char2idx[song] for song in string ])
  return vectorized_songs
vectorized_songs = vectorize_string(songs_joined)

In [ ]:
print ('{} ---- characters mapped to int ----> {}'.format(repr(songs_joined[:10]), vectorized_songs[:10]))
# check that vectorized_songs is a numpy array
assert isinstance(vectorized_songs, np.ndarray), "returned result should be a numpy array"

### Create training examples and targets

In [ ]:
### Batch definition to create training examples ###

def get_batch(vectorized_songs, seq_length, batch_size):
  # the length of the vectorized songs string
  n = vectorized_songs.shape[0] - 1
  # randomly choose the starting indices for the examples in the training batch
  idx = np.random.choice(n-seq_length, batch_size)

  '''TODO: construct a list of input sequences for the training batch'''
  input_batch = [vectorized_songs[i:i+seq_length] for i in idx]
  '''TODO: construct a list of output sequences for the training batch'''
  output_batch = [vectorized_songs[i+1:i+seq_length+1] for i in idx]

  # x_batch, y_batch provide the true inputs and targets for network training
  x_batch = np.reshape(input_batch, [batch_size, seq_length])
  y_batch = np.reshape(output_batch, [batch_size, seq_length])
  return x_batch, y_batch


# Perform some simple tests to make sure your batch function is working properly!
test_args = (vectorized_songs, 10, 2)
if not mdl.lab1.test_batch_func_types(get_batch, test_args) or \
   not mdl.lab1.test_batch_func_shapes(get_batch, test_args) or \
   not mdl.lab1.test_batch_func_next_step(get_batch, test_args):
   print("======\n[FAIL] could not pass tests")
else:
   print("======\n[PASS] passed all tests!")

In [ ]:
x_batch, y_batch = get_batch(vectorized_songs, seq_length=5, batch_size=1)

for i, (input_idx, target_idx) in enumerate(zip(np.squeeze(x_batch), np.squeeze(y_batch))):
    print("Step {:3d}".format(i))
    print("  input: {} ({:s})".format(input_idx, repr(idx2char[input_idx])))
    print("  expected output: {} ({:s})".format(target_idx, repr(idx2char[target_idx])))

## 1.4 Define the Generative Adversarial Networks (GAN) model

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Reshape, Input, Conv1D, Flatten, TimeDistributed
from tensorflow.keras.layers import LSTM, Bidirectional, LeakyReLU, BatchNormalization
from tensorflow.keras.optimizers import Adam

# Define constants
latent_dim = 100  # Dimension of the noise vector
sequence_length = 256  # Longer sequence for music patterns
features = 128  # Number of music features (e.g., MIDI notes, velocity, etc.)

# Generator Model with LSTM layers
def build_generator():
    model = Sequential([
        Input(shape=(latent_dim,)),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dense(sequence_length * 64, activation='relu'),
        BatchNormalization(),
        Reshape((sequence_length, 64)),

        # Adding LSTM layers for temporal understanding
        LSTM(256, return_sequences=True),
        Dropout(0.3),

        # Bidirectional LSTM can capture patterns in both directions
        Bidirectional(LSTM(256, return_sequences=True)),
        Dropout(0.3),

        # Final layers to output the music representation
        TimeDistributed(Dense(128, activation='relu')),
        TimeDistributed(Dense(features, activation='tanh'))  # tanh for normalized output
    ])
    return model

# Discriminator Model with LSTM layers
def build_discriminator():
    model = Sequential([
        Input(shape=(sequence_length, features)),

        # Convolutional layers to detect patterns
        Conv1D(64, kernel_size=5, strides=2, padding='same'),
        LeakyReLU(alpha=0.2),
        Dropout(0.3),

        Conv1D(128, kernel_size=5, strides=2, padding='same'),
        LeakyReLU(alpha=0.2),
        Dropout(0.3),

        # LSTM layers to capture temporal patterns
        LSTM(256, return_sequences=True),
        Dropout(0.3),

        Flatten(),
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    return model

# Create the generator and discriminator
generator = build_generator()
discriminator = build_discriminator()

# Compile the discriminator
discriminator.compile(
    optimizer=Adam(learning_rate=0.0001, beta_1=0.5),  # Lower learning rate, adjusted beta
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Combined GAN Model
discriminator.trainable = False
gan_input = Input(shape=(latent_dim,))
generated_sequence = generator(gan_input)
gan_output = discriminator(generated_sequence)
gan_model = Model(gan_input, gan_output)

# Compile the GAN model
gan_model.compile(
    optimizer=Adam(learning_rate=0.0001, beta_1=0.5),
    loss='binary_crossentropy'
)

# Summarize the models
generator.summary()
discriminator.summary()
gan_model.summary()

### Test out the GAN model

In [ ]:
# Generate a batch of random noise inputs
noise_dim = 100
x = np.random.normal(0, 1, (32, noise_dim)).astype(np.float32)  # Convert to float32

# The following line is not needed as it creates a separate Reshape layer
# tf.keras.layers.Reshape((5, 83))  # Assuming you want a shape of (32, 5, 83)

# Generate sequences using the generator
x_gen = generator(x)

# Print the shapes
print("Noise input shape:  ", x.shape, " # (batch_size, noise_dim)")
print("Generated shape:    ", x_gen.shape, " # (batch_size, sequence_length, vocab_size)")

### Predictions from the untrained model

In [ ]:
# Generate a batch of random noise inputs
noise_dim = 100
x = np.random.normal(0, 1, (1, noise_dim))  # Single noise vector input for prediction

# Generate a sequence from the generator
pred_tensor = generator.predict(x)  # pred_tensor shape: (1, seq_length, vocab_size)

# Reshape pred_tensor to 2D for tf.random.categorical
pred_tensor_2d = tf.reshape(pred_tensor, [-1, pred_tensor.shape[-1]])  # Shape becomes (seq_length, vocab_size)

# Sample indices from the probability distribution of predicted tokens
sampled_indices = tf.random.categorical(pred_tensor_2d, num_samples=1)
sampled_indices = tf.squeeze(sampled_indices, axis=-1).numpy()

# Print the sampled indices as characters
print("Sampled Indices:\n", sampled_indices)

# Convert the indices back to characters
predicted_chars = "".join(idx2char[sampled_indices])
print("Generated Sequence: \n", repr(predicted_chars))


In [ ]:
print("Input Noise Vector: \n", repr(x[0])) # Print the noise vector instead of trying to convert it to characters
print()
# Ensure indices are within the valid range for idx2char
valid_indices = [idx % len(idx2char) for idx in sampled_indices]
print("Next Char Predictions: \n", repr("".join(idx2char[valid_indices])))

## 1.5 Training the model: loss and training operations

In [ ]:
### Defining the loss function ###
def compute_loss(labels, logits):
    loss = tf.keras.losses.BinaryCrossentropy(from_logits=True)(labels, logits)
    return loss

'''TODO: compute the loss using the true next characters from the example batch
    and the predictions from the untrained model several cells above'''
example_batch_loss = compute_loss(tf.ones_like(x_gen), x_gen)  # Use BinaryCrossentropy for GAN

print("Generated shape: ", x_gen.shape, " # (batch_size, sequence_length, vocab_size)")
print("scalar_loss:      ", example_batch_loss.numpy())

In [ ]:
import tensorflow as tf
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm import tqdm

# -----------------------------
# Configuration & Hyperparameters
# -----------------------------
tf.random.set_seed(42)
np.random.seed(42)

latent_dim = 100
sequence_length = 100
vocab_size = 83  # Number of unique tokens in your ABC dataset
batch_size = 64
epochs = 50000
learning_rate = 1e-3  # Slightly lower learning rate for better stability
learning_rate_decay = 0.999  # Add learning rate decay

# Directory structure
checkpoint_dir = './music_gan_checkpoints/'
sample_dir = './music_samples/'
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(sample_dir, exist_ok=True)
checkpoint_prefix_gen = os.path.join(checkpoint_dir, "generator")
checkpoint_prefix_disc = os.path.join(checkpoint_dir, "discriminator")

# -----------------------------
# Dataset Preparation
# -----------------------------
def create_music_dataset(data, seq_length=100, batch_size=64, vocab_size=83):
    n = len(data)
    inputs = [data[i:i+seq_length] for i in range(0, n - seq_length, seq_length//2)]  # Added overlap
    one_hot_inputs = tf.one_hot(inputs, depth=vocab_size)
    dataset = tf.data.Dataset.from_tensor_slices(one_hot_inputs)
    return dataset.shuffle(10000).batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

# Using your existing vectorized_songs data directly
dataset = create_music_dataset(vectorized_songs, sequence_length, batch_size, vocab_size)

# -----------------------------
# Generator - Enhanced with residual connections
# -----------------------------
def build_generator():
    noise_input = tf.keras.layers.Input(shape=(latent_dim,))

    # Initial dense and reshape
    x = tf.keras.layers.Dense(256 * sequence_length, activation='relu')(noise_input)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Reshape((sequence_length, 256))(x)

    # Convolutional blocks with residual connections
    skip1 = x  # First skip connection

    # First conv block
    x = tf.keras.layers.Conv1D(256, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    # Second conv block
    x = tf.keras.layers.Conv1D(256, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    # Add skip connection
    x = tf.keras.layers.add([x, skip1])

    # Third conv block
    skip2 = x  # Second skip connection
    x = tf.keras.layers.Conv1D(128, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    # Fourth conv block
    x = tf.keras.layers.Conv1D(128, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    # Add skip connection (with projection if needed)
    skip2_projection = tf.keras.layers.Conv1D(128, 1, padding='same')(skip2)
    x = tf.keras.layers.add([x, skip2_projection])

    # Output layer
    output = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(vocab_size, activation='softmax'))(x)

    return tf.keras.models.Model(noise_input, output)

# -----------------------------
# Discriminator - Enhanced architecture
# -----------------------------
def build_discriminator():
    inputs = tf.keras.layers.Input(shape=(sequence_length, vocab_size))

    # First conv block
    x = tf.keras.layers.Conv1D(128, 5, padding='same')(inputs)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    # Second conv block
    x = tf.keras.layers.Conv1D(256, 5, strides=2, padding='same')(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    # Third conv block
    x = tf.keras.layers.Conv1D(512, 5, strides=2, padding='same')(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    # Flatten and dense layers
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    return tf.keras.models.Model(inputs, output)

generator = build_generator()
discriminator = build_discriminator()

# Print model summaries
generator.summary()
discriminator.summary()

# -----------------------------
# Losses & Optimizers with learning rate schedulers
# -----------------------------
bce = tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1)  # Added label smoothing

# Learning rate schedulers
lr_schedule_g = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=learning_rate,
    decay_steps=1000,
    decay_rate=learning_rate_decay
)

lr_schedule_d = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=learning_rate,
    decay_steps=1000,
    decay_rate=learning_rate_decay
)

g_opt = tf.keras.optimizers.Adam(lr_schedule_g, beta_1=0.5)
d_opt = tf.keras.optimizers.Adam(lr_schedule_d, beta_1=0.5)

# -----------------------------
# Helper function to generate music samples
# -----------------------------
def generate_samples(epoch, num_samples=4):
    noise = tf.random.normal([num_samples, latent_dim])
    generated_samples = generator(noise, training=False)

    # Convert from one-hot to indices
    sample_indices = tf.argmax(generated_samples, axis=-1).numpy()

    # Save samples for later conversion
    np.save(f"{sample_dir}/sample_epoch_{epoch}.npy", sample_indices)
    return sample_indices

# -----------------------------
# Training Step with Wasserstein loss characteristics
# -----------------------------
@tf.function
def train_step(real_batch):
    batch_size = tf.shape(real_batch)[0]
    noise = tf.random.normal([batch_size, latent_dim])

    # Add some noise to labels for label smoothing (helps with GAN stability)
    real_labels = tf.random.uniform([batch_size, 1], 0.8, 1.0)
    fake_labels = tf.random.uniform([batch_size, 1], 0.0, 0.2)

    with tf.GradientTape() as g_tape, tf.GradientTape() as d_tape:
        # Generate fake samples
        fake_batch = generator(noise, training=True)

        # Get discriminator outputs
        real_output = discriminator(real_batch, training=True)
        fake_output = discriminator(fake_batch, training=True)

        # Calculate losses
        d_loss_real = bce(real_labels, real_output)
        d_loss_fake = bce(fake_labels, fake_output)
        d_loss = d_loss_real + d_loss_fake

        # Generator loss
        g_loss = bce(tf.ones_like(fake_output), fake_output)

    # Calculate gradients
    g_grads = g_tape.gradient(g_loss, generator.trainable_variables)
    d_grads = d_tape.gradient(d_loss, discriminator.trainable_variables)

    # Clip gradients for stability (optional)
    g_grads, _ = tf.clip_by_global_norm(g_grads, 1.0)
    d_grads, _ = tf.clip_by_global_norm(d_grads, 1.0)

    # Apply gradients
    g_opt.apply_gradients(zip(g_grads, generator.trainable_variables))
    d_opt.apply_gradients(zip(d_grads, discriminator.trainable_variables))

    return g_loss, d_loss

# -----------------------------
# Training Loop with Improved Logging & Monitoring
# -----------------------------
g_losses = []
d_losses = []

pbar = tqdm(total=epochs)
train_iter = iter(dataset.repeat())

for step in range(epochs):
    real_batch = next(train_iter)
    g_loss, d_loss = train_step(real_batch)

    g_losses.append(g_loss.numpy())
    d_losses.append(d_loss.numpy())

    # Update progress bar with current losses
    if step % 50 == 0:
        pbar.set_description(f"G_loss: {g_loss:.4f}, D_loss: {d_loss:.4f}")
    pbar.update(1)

    # Generate samples and checkpoints every 500 steps
    if step % 500 == 0:
        # Save model weights
        generator.save_weights(f"{checkpoint_prefix_gen}_{step}.weights.h5")
        discriminator.save_weights(f"{checkpoint_prefix_disc}_{step}.weights.h5")

        # Generate and save samples
        generate_samples(step)

        # Save losses plot
        if step > 0:
            plt.figure(figsize=(10, 5))
            plt.plot(g_losses, label="Generator Loss", alpha=0.7)
            plt.plot(d_losses, label="Discriminator Loss", alpha=0.7)
            plt.xlabel("Training Step")
            plt.ylabel("Loss")
            plt.title("GAN Training Losses")
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.6)
            plt.tight_layout()
            plt.savefig(f"{checkpoint_dir}/losses_step_{step}.png")
            plt.close()

pbar.close()

# -----------------------------
# Final Loss Plot
# -----------------------------
plt.figure(figsize=(12, 6))
plt.plot(g_losses, label="Generator Loss", alpha=0.7)
plt.plot(d_losses, label="Discriminator Loss", alpha=0.7)
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("GAN Training Losses")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("gan_training_losses_final.png")
plt.show()

# -----------------------------
# Generate final samples
# -----------------------------
final_samples = generate_samples(epochs, num_samples=10)
print("Training complete! Final samples saved to:", sample_dir)

## 1.6 Evaluation of Model
**Calculation of Accuracy**

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

def evaluate_music_gan_accuracy(generator, discriminator, latent_dim, real_music_batch, save_dir=None):
    """
    Evaluate the accuracy of a music generation GAN.

    Parameters:
    - generator: The generator model
    - discriminator: The discriminator model
    - latent_dim: Dimension of the noise input
    - real_music_batch: A batch of real music sequences
    - save_dir: Directory to save evaluation results (optional)

    Returns:
    - Dictionary of accuracy metrics
    """
    batch_size = real_music_batch.shape[0]

    # Generate fake music sequences
    random_noise = tf.random.normal([batch_size, latent_dim])
    generated_music = generator(random_noise, training=False)

    # Get discriminator predictions
    real_labels = tf.ones((batch_size, 1))
    fake_labels = tf.zeros((batch_size, 1))

    real_output = discriminator(real_music_batch, training=False)
    fake_output = discriminator(generated_music, training=False)

    # Calculate discriminator accuracy
    real_predictions = tf.cast(real_output > 0.5, tf.float32)
    fake_predictions = tf.cast(fake_output > 0.5, tf.float32)

    real_accuracy = tf.reduce_mean(tf.cast(tf.equal(real_predictions, real_labels), tf.float32))
    fake_accuracy = tf.reduce_mean(tf.cast(tf.equal(fake_predictions, fake_labels), tf.float32))
    overall_accuracy = (real_accuracy + fake_accuracy) / 2.0

    # Generator fooling rate (how often the generator fools the discriminator)
    generator_fooling_rate = tf.reduce_mean(tf.cast(fake_output > 0.5, tf.float32))

    # Print results
    print(f"==== GAN Accuracy Evaluation ====")
    print(f"Discriminator Real Accuracy: {real_accuracy.numpy():.4f}")
    print(f"Discriminator Fake Accuracy: {fake_accuracy.numpy():.4f}")
    print(f"Discriminator Overall Accuracy: {overall_accuracy.numpy():.4f}")
    print(f"Generator Fooling Rate: {generator_fooling_rate.numpy():.4f}")

    # Save results if a directory is provided
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

        # Create a simple bar chart of accuracies
        plt.figure(figsize=(10, 6))
        metrics = ['Real Acc', 'Fake Acc', 'Overall Acc', 'Fooling Rate']
        values = [real_accuracy.numpy(), fake_accuracy.numpy(),
                 overall_accuracy.numpy(), generator_fooling_rate.numpy()]

        bars = plt.bar(metrics, values, color=['blue', 'orange', 'green', 'red'])

        # Add value labels on top of the bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{height:.4f}', ha='center', va='bottom')

        plt.ylim(0, 1.1)  # Accuracy range from 0 to 1
        plt.title('Music GAN Accuracy Metrics')
        plt.ylabel('Accuracy')
        plt.grid(axis='y', alpha=0.3)
        plt.savefig(f"{save_dir}/gan_accuracy.png")
        plt.close()

    # Return metrics as a dictionary
    return {
        "real_accuracy": real_accuracy.numpy(),
        "fake_accuracy": fake_accuracy.numpy(),
        "overall_accuracy": overall_accuracy.numpy(),
        "generator_fooling_rate": generator_fooling_rate.numpy()
    }

def track_accuracy_history(history, metrics, iteration):
    """Track accuracy metrics over time"""
    if not history:
        history = {
            "iterations": [],
            "real_accuracy": [],
            "fake_accuracy": [],
            "overall_accuracy": [],
            "generator_fooling_rate": []
        }

    history["iterations"].append(iteration)
    history["real_accuracy"].append(metrics["real_accuracy"])
    history["fake_accuracy"].append(metrics["fake_accuracy"])
    history["overall_accuracy"].append(metrics["overall_accuracy"])
    history["generator_fooling_rate"].append(metrics["generator_fooling_rate"])

    return history

def plot_accuracy_history(history, save_dir):
    """Plot the history of accuracy metrics"""
    if not history or len(history["iterations"]) < 2:
        print("Not enough data points to plot history")
        return

    plt.figure(figsize=(12, 6))
    plt.plot(history["iterations"], history["real_accuracy"],
             label="Real Accuracy", marker='o', markersize=4)
    plt.plot(history["iterations"], history["fake_accuracy"],
             label="Fake Accuracy", marker='s', markersize=4)
    plt.plot(history["iterations"], history["overall_accuracy"],
             label="Overall Accuracy", marker='^', markersize=4)
    plt.plot(history["iterations"], history["generator_fooling_rate"],
             label="Generator Fooling Rate", marker='*', markersize=4)

    plt.xlabel("Training Iterations")
    plt.ylabel("Accuracy")
    plt.title("Music GAN Accuracy Over Time")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f"{save_dir}/accuracy_history.png")
    plt.close()

# Example usage:
# Create directory for results
eval_dir = "./music_gan_evaluation/"
os.makedirs(eval_dir, exist_ok=True)

# Initialize history tracking
accuracy_history = {}

# For demonstration - evaluate once
real_music_batch = next(train_iter)  # Get a batch of real music data
metrics = evaluate_music_gan_accuracy(generator, discriminator, latent_dim, real_music_batch, eval_dir)
accuracy_history = track_accuracy_history(accuracy_history, metrics, 0)



**Calculate Avg Loss, Note Accuracy, Chord Accuracy and Perplexity**

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------- #
# Define Generator Model (Conv1D, No RNN)
# --------------------------------------------- #
latent_dim = 100
vocab_size = 83
sequence_length = 100

def build_generator():
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(latent_dim,)),
        tf.keras.layers.Dense(256 * sequence_length, activation='relu'),
        tf.keras.layers.Reshape((sequence_length, 256)),
        tf.keras.layers.Conv1D(128, kernel_size=3, padding='same', activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Conv1D(128, kernel_size=3, padding='same', activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(vocab_size, activation='softmax'))  # Critical
    ])

# Instantiate generator
generator = build_generator()

# --------------------------------------------- #
# Dataset Creation from Real Data
# --------------------------------------------- #

def create_dataset(data, seq_length=100, batch_size=32, vocab_size=83):
    n = len(data) - 1
    inputs = [data[i:i+seq_length] for i in range(0, n - seq_length, seq_length)]
    one_hot_inputs = tf.one_hot(inputs, depth=vocab_size)
    dataset = tf.data.Dataset.from_tensor_slices(one_hot_inputs)
    return dataset.batch(batch_size)

# --------------------------------------------- #
# Modified Evaluation Metrics for Music GAN
# --------------------------------------------- #

def evaluate_generator(generator, dataset, latent_dim):
    """
    Evaluate a music GAN generator with adapted loss, accuracy, and perplexity metrics

    Parameters:
    - generator: The generator model
    - dataset: Dataset containing real music samples
    - latent_dim: Dimension of the noise vector

    Returns:
    - Metrics dictionary
    """
    # Statistics to track
    losses = []
    accuracies = []
    perplexities = []

    # Also track distribution-based metrics
    real_note_dist = np.zeros(vocab_size)
    gen_note_dist = np.zeros(vocab_size)

    for real_batch in dataset:
        batch_size = tf.shape(real_batch)[0]

        # Generate music from random noise
        noise = tf.random.normal([batch_size, latent_dim])
        generated_music = generator(noise, training=False)

        # ---------- ADAPTED METRICS FOR GAN EVALUATION ----------

        # 1. Modified Loss: Use cross-entropy between generated distribution and empirical real distribution
        # This measures how well the generator captures the overall note distribution in the real data

        # Get distributions for this batch
        real_indices = tf.argmax(real_batch, axis=-1).numpy()
        real_batch_dist = np.bincount(real_indices.flatten(), minlength=vocab_size)
        real_batch_dist = real_batch_dist / np.sum(real_batch_dist)

        gen_indices = tf.argmax(generated_music, axis=-1).numpy()
        gen_batch_dist = np.bincount(gen_indices.flatten(), minlength=vocab_size)
        gen_batch_dist = gen_batch_dist / np.sum(gen_batch_dist)

        # Distribution loss (cross-entropy)
        epsilon = 1e-10  # prevent log(0)
        dist_loss = -np.sum(real_batch_dist * np.log(gen_batch_dist + epsilon))
        losses.append(dist_loss)

        # Update overall distributions
        real_note_dist += real_batch_dist
        gen_note_dist += gen_batch_dist

        # 2. Modified Accuracy: Measure alignment of statistical properties
        # Instead of direct token matching, we compare statistical properties

        # Use n-gram distribution accuracy (for n=2, measure transitions)
        real_transitions = {}
        gen_transitions = {}

        # Extract transitions from real data
        for seq in real_indices:
            for i in range(len(seq)-1):
                pair = (seq[i], seq[i+1])
                real_transitions[pair] = real_transitions.get(pair, 0) + 1

        # Extract transitions from generated data
        for seq in gen_indices:
            for i in range(len(seq)-1):
                pair = (seq[i], seq[i+1])
                gen_transitions[pair] = gen_transitions.get(pair, 0) + 1

        # Calculate transition accuracy (shared transitions / total unique transitions)
        shared_transitions = set(real_transitions.keys()).intersection(set(gen_transitions.keys()))
        total_transitions = set(real_transitions.keys()).union(set(gen_transitions.keys()))

        if len(total_transitions) > 0:
            transition_accuracy = len(shared_transitions) / len(total_transitions)
        else:
            transition_accuracy = 0.0

        accuracies.append(transition_accuracy)

        # 3. Modified Perplexity: Measures "surprise" using the real data distribution
        # Lower is better - measures how well the generated distribution matches expected patterns

        # Calculate n-gram frequencies from real data
        note_counts = {}
        bigram_counts = {}

        for seq in real_indices:
            for i in range(len(seq)):
                note = seq[i]
                note_counts[note] = note_counts.get(note, 0) + 1

                if i < len(seq) - 1:
                    bigram = (seq[i], seq[i+1])
                    bigram_counts[bigram] = bigram_counts.get(bigram, 0) + 1

        # Convert to probabilities
        total_notes = sum(note_counts.values())
        total_bigrams = sum(bigram_counts.values())

        # Evaluate perplexity on generated sequences
        log_likelihood = 0.0
        token_count = 0

        for seq in gen_indices:
            for i in range(len(seq)-1):
                bigram = (seq[i], seq[i+1])
                # Smooth counts for unseen bigrams
                bigram_count = bigram_counts.get(bigram, 0) + 0.1
                unigram_count = note_counts.get(seq[i], 0) + 0.1

                # P(next_note | current_note)
                prob = bigram_count / (unigram_count + 0.1 * vocab_size)
                log_likelihood += np.log(prob)
                token_count += 1

        # Perplexity = exp(-avg_log_likelihood)
        if token_count > 0:
            perplexity = np.exp(-log_likelihood / token_count)
        else:
            perplexity = float('inf')

        perplexities.append(perplexity)

    # Calculate overall metrics
    avg_loss = np.mean(losses)
    avg_accuracy = np.mean(accuracies)
    avg_perplexity = np.mean(perplexities)

    # Also calculate JS divergence between overall distributions
    real_note_dist = real_note_dist / np.sum(real_note_dist) if np.sum(real_note_dist) > 0 else real_note_dist
    gen_note_dist = gen_note_dist / np.sum(gen_note_dist) if np.sum(gen_note_dist) > 0 else gen_note_dist

    # Calculate JS divergence
    m_dist = 0.5 * (real_note_dist + gen_note_dist)
    js_div = 0.5 * (
        np.sum(real_note_dist * np.log(real_note_dist/(m_dist+1e-10) + 1e-10)) +
        np.sum(gen_note_dist * np.log(gen_note_dist/(m_dist+1e-10) + 1e-10))
    )

    # Convert to similarity (1 = identical, 0 = completely different)
    distribution_similarity = 1 / (1 + js_div)

    # Print results
    print(f"\n GAN Generator Evaluation Results:")
    print(f"- Distribution Loss:      {avg_loss:.4f} (lower is better)")
    print(f"- Transition Accuracy:    {avg_accuracy:.4f} (higher is better)")
    print(f"- Perplexity:             {avg_perplexity:.4f} (lower is better)")
    print(f"- Distribution Similarity: {distribution_similarity:.4f} (higher is better)")

    return {
        "loss": avg_loss,
        "accuracy": avg_accuracy,
        "perplexity": avg_perplexity,
        "distribution_similarity": distribution_similarity
    }

# Run evaluation
real_dataset = create_dataset(vectorized_songs, seq_length=100, batch_size=32, vocab_size=83)
metrics = evaluate_generator(generator, real_dataset, latent_dim=100)

## 1.7 Generate music using the GAN model

### Restore the latest checkpoint

In [ ]:
import tensorflow as tf
import os

# Define constants
latent_dim = 100
vocab_size = 83
sequence_length = 100

# Function to rebuild the Generator model
def build_generator():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(latent_dim,)),
        tf.keras.layers.Dense(256 * sequence_length, activation='relu'),  # Increase the output dimension of the Dense layer
        tf.keras.layers.Reshape((sequence_length, 256)),  # Reshape to (sequence_length, 256)
        tf.keras.layers.LSTM(128, return_sequences=True),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.LSTM(128, return_sequences=True),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(vocab_size, activation='softmax')
    ])
    return model

# Function to rebuild the Discriminator model
def build_discriminator():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(sequence_length, vocab_size)),
        tf.keras.layers.LSTM(128, return_sequences=True),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.LSTM(128),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    return model

# Create the generator and discriminator
generator = build_generator()
discriminator = build_discriminator()

# Compile the discriminator
discriminator.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002), loss='binary_crossentropy', metrics=['accuracy'])

# Freeze discriminator weights for the combined model
discriminator.trainable = False

# Combined GAN Model
gan_input = tf.keras.layers.Input(shape=(latent_dim,))
generated_sequence = generator(gan_input)
gan_output = discriminator(generated_sequence)
gan_model = tf.keras.models.Model(gan_input, gan_output)

# Compile the GAN model
gan_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002), loss='binary_crossentropy')

# Define checkpoint directories for Generator and Discriminator
generator_checkpoint_dir = './checkpoints/generator/'
discriminator_checkpoint_dir = './checkpoints/discriminator/'

# Function to restore model from checkpoints
def restore_model_from_checkpoint(model, checkpoint_dir, model_name):
    print(f"Checking for checkpoints in: {checkpoint_dir}")
    if os.path.exists(checkpoint_dir):
        checkpoint_files = [f for f in os.listdir(checkpoint_dir) if f.startswith('my_ckpt')]
        if checkpoint_files:
            print(f"Found checkpoint files for {model_name}: {checkpoint_files}")
            # Restore the model weights from the last checkpoint
            latest_checkpoint = tf.train.latest_checkpoint(checkpoint_dir)
            if latest_checkpoint:
                model.load_weights(latest_checkpoint)
                print(f"Loaded weights from {latest_checkpoint} for {model_name}")
                # Build the model with batch_size=1
                model.build(tf.TensorShape([1, None, None]))  # Adjust input shape as needed
                model.summary()
            else:
                print(f"No checkpoint files found for {model_name} in the directory.")
        else:
            print(f"No checkpoint files found for {model_name} in the directory.")
    else:
        print(f"Checkpoint directory for {model_name} not found.")

# Restore Generator and Discriminator models
restore_model_from_checkpoint(generator, generator_checkpoint_dir, "Generator")
restore_model_from_checkpoint(discriminator, discriminator_checkpoint_dir, "Discriminator")

# Restore the GAN model if needed (if you saved checkpoints for the GAN model separately)
# restore_model_from_checkpoint(gan_model, gan_checkpoint_dir, "GAN")


In [ ]:
import tensorflow as tf
import numpy as np
import os
import re
import random  # Add this import
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.display import Audio, display

def main():
    print(" Loading dataset and GAN model...")
    import mitdeeplearning as mdl
    songs = mdl.lab1.load_training_data()

    vocab = sorted(set("".join(songs)))
    char2idx = {u: i for i, u in enumerate(vocab)}
    idx2char = np.array(vocab)

    # Load GAN model
    try:
        latent_dim = 100
        sequence_length = 100
        vocab_size = len(vocab)

        # Load the generator model
        generator = build_generator()
        generator.load_weights("/content/music_gan_checkpoints/generator_49500.weights.h5")
        print(" GAN generator model loaded successfully.")

    except Exception as e:
        print(f" Failed to load GAN model: {e}")
        return

    # Pick original song
    original_song = random.choice(songs)
    original_length = len(original_song)

    print(f"\n Selected original song ({original_length} characters):")
    print(original_song[:200] + "...")

    # 1. GENERATED: Create a completely new song using GAN
    print(f"\n Creating GENERATED song (new composition)...")
    generated_song = generate_long_song_with_gan(
        generator, latent_dim, idx2char, target_length=original_length * 2, sequence_length=sequence_length
    )

    # 2. RECONSTRUCTED: Attempt to recreate the selected original song
    print(f"\n Creating RECONSTRUCTED song (recreating selected song)...")
    reconstructed_song = reconstruct_original_song_with_gan(
        generator, original_song, char2idx, idx2char, latent_dim, sequence_length
    )

    # Fix both for audio conversion
    generated_song = fix_abc_for_audio(generated_song)
    reconstructed_song = fix_abc_for_audio(reconstructed_song)

    # Save files with clear names
    with open("1_original_song.abc", "w") as f:
        f.write(original_song)
    with open("2_generated_song.abc", "w") as f:
        f.write(generated_song)
    with open("3_reconstructed_song.abc", "w") as f:
        f.write(reconstructed_song)

    # Length comparison
    print(f"\n Song comparison:")
    print(f"ORIGINAL:      {len(original_song):4d} chars")
    print(f"GENERATED:     {len(generated_song):4d} chars")
    print(f"RECONSTRUCTED: {len(reconstructed_song):4d} chars")

    # Count notes
    orig_notes = len(re.findall(r'[A-Ga-g]', original_song))
    gen_notes = len(re.findall(r'[A-Ga-g]', generated_song))
    rec_notes = len(re.findall(r'[A-Ga-g]', reconstructed_song))

    print(f"ORIGINAL notes:      {orig_notes}")
    print(f"GENERATED notes:     {gen_notes}")
    print(f"RECONSTRUCTED notes: {rec_notes}")

    # Convert to audio
    print("\n Converting to audio...")
    original_audio = convert_abc_to_audio(original_song, "1_original_song")
    generated_audio = convert_abc_to_audio(generated_song, "2_generated_song")
    reconstructed_audio = convert_abc_to_audio(reconstructed_song, "3_reconstructed_song")

    # Play audio with clear descriptions
    print("\n🎧 AUDIO COMPARISON:")

    if original_audio:
        print(f"\n 1. ORIGINAL (the selected song):")
        display(Audio(filename=original_audio))

    if generated_audio:
        print(f"\n 2. GENERATED (new GAN composition):")
        display(Audio(filename=generated_audio))

    if reconstructed_audio:
        print(f"\n 3. RECONSTRUCTED (GAN trying to recreate the original):")
        display(Audio(filename=reconstructed_audio))

    print("\n Compare how well the GAN reconstructed vs. generated new music!")

def build_generator():
    """Build the generator model with identical architecture to the training model"""
    latent_dim = 100
    sequence_length = 100
    vocab_size = 83

    noise_input = tf.keras.layers.Input(shape=(latent_dim,))

    # Initial dense and reshape
    x = tf.keras.layers.Dense(256 * sequence_length, activation='relu')(noise_input)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Reshape((sequence_length, 256))(x)

    # Convolutional blocks with residual connections
    skip1 = x  # First skip connection

    # First conv block
    x = tf.keras.layers.Conv1D(256, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    # Second conv block
    x = tf.keras.layers.Conv1D(256, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    # Add skip connection
    x = tf.keras.layers.add([x, skip1])

    # Third conv block
    skip2 = x  # Second skip connection
    x = tf.keras.layers.Conv1D(128, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    # Fourth conv block
    x = tf.keras.layers.Conv1D(128, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    # Add skip connection (with projection if needed)
    skip2_projection = tf.keras.layers.Conv1D(128, 1, padding='same')(skip2)
    x = tf.keras.layers.add([x, skip2_projection])

    # Output layer
    output = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(vocab_size, activation='softmax'))(x)

    return tf.keras.models.Model(noise_input, output)

# === Function to Reconstruct the Original Song with GAN ===
def reconstruct_original_song_with_gan(generator, original_song, char2idx, idx2char, latent_dim=100, seq_length=100):
    """Direct reconstruction that preserves original musical structure using GAN"""
    print(" Reconstructing the original song through GAN...")

    # Extract header to preserve it exactly
    header_match = re.search(r'(X:.*?K:[^\n]+\n)', original_song, re.DOTALL)
    if header_match:
        header = header_match.group(1)
        body = original_song[len(header):]
    else:
        header = "X:1\nT:Reconstructed\nM:4/4\nL:1/8\nK:C\n"
        body = original_song

    # Add tempo if not present in header
    if "Q:" not in header:
        header = re.sub(r'(K:[^\n]+\n)', r'\1Q:1/4=120\n', header)

    # Extract the musical structure from the original - bar lines and repeated sections
    structure_template = re.sub(r'[A-Ga-g,\'0-9\^\_\=]', 'x', body)  # Replace notes with x

    # Process the body in small chunks with focus on musical motifs
    original_notes = re.findall(r'[A-Ga-g][,\'0-9]*', body)
    original_bars = body.split('|')

    print(f" Original song has {len(original_notes)} notes across {len(original_bars)} bars")

    # Direct approach: Use the GAN to generate each bar
    reconstructed_bars = []

    # For each bar in the original
    for i, bar in enumerate(tqdm(original_bars, desc="Processing bars")):
        if not bar.strip():  # Skip empty bars
            continue

        # Find notes in this bar
        bar_notes = re.findall(r'[A-Ga-g][,\'0-9]*', bar)
        if not bar_notes:  # Skip bars without notes
            reconstructed_bars.append(bar)
            continue

        # Generate random noise as input to GAN
        noise = tf.random.normal([1, latent_dim])

        # Generate sequence with GAN
        generated_logits = generator(noise, training=False)

        # Convert logits to indices (most likely character at each position)
        generated_indices = tf.argmax(generated_logits[0], axis=-1).numpy()

        # Convert indices to characters
        generated_chars = [idx2char[idx] for idx in generated_indices]
        generated_text = ''.join(generated_chars)

        # Extract notes from generated text
        generated_notes = re.findall(r'[A-Ga-g][,\'0-9]*', generated_text)

        # If no notes generated, use some from the original
        if not generated_notes and bar_notes:
            generated_notes = bar_notes

        # Recreate bar structure with generated notes
        bar_template = re.sub(r'[A-Ga-g][,\'0-9]*', 'NOTE', bar)
        reconstructed_bar = bar_template

        # Replace NOTE placeholders with actual notes
        for j, note in enumerate(generated_notes):
            if 'NOTE' in reconstructed_bar:
                reconstructed_bar = reconstructed_bar.replace('NOTE', note, 1)
            else:
                reconstructed_bar += note + " "

        # Ensure bar ends with |
        if not reconstructed_bar.endswith('|'):
            reconstructed_bar += ' |'

        reconstructed_bars.append(reconstructed_bar)

    # Combine bars
    reconstructed_body = ''.join(reconstructed_bars)

    # Check note count
    original_note_count = len(re.findall(r'[A-Ga-g]', body))
    reconstructed_note_count = len(re.findall(r'[A-Ga-g]', reconstructed_body))

    # If not enough notes, supplement with original material
    if reconstructed_note_count < original_note_count * 0.7:
        print(f" Still insufficient notes ({reconstructed_note_count} vs {original_note_count})")
        print(" Adding original motifs to enhance reconstruction")

        # Extract musical phrases from original
        phrases = []
        for bar in original_bars:
            if len(bar.strip()) > 3:  # Non-empty bar
                phrases.append(bar)

        # Add some original phrases
        sample_count = max(1, (original_note_count - reconstructed_note_count) // 20)
        selected_phrases = random.sample(phrases, min(sample_count, len(phrases)))

        for phrase in selected_phrases:
            reconstructed_body += phrase

    # Final processing
    reconstructed_body = fix_music_structure(reconstructed_body)

    # Complete song
    result = header + reconstructed_body

    # Final note count
    final_note_count = len(re.findall(r'[A-Ga-g]', result))

    print(f"Reconstructed {len(result)} characters (original was {len(original_song)})")
    print(f"Original notes: {original_note_count}, Reconstructed notes: {final_note_count}")

    return result

def fix_music_structure(music_text):
    """Fix common ABC notation structural issues"""
    # Ensure bar lines are properly spaced
    music_text = re.sub(r'\|\s*\|', '|', music_text)

    # Remove invalid characters often appearing in generated music
    music_text = re.sub(r'[^ABCDEFGabcdefgz0-9\|\[\]\/\(\)\^\=\_\,\'\~\:\-\+ ]', '', music_text)

    # Ensure proper bar line spacing
    music_text = re.sub(r'([A-Ga-g][0-9]?[,\']*)(?!\||\s)', r'\1 ', music_text)

    # Fix trailing bars
    if not music_text.endswith('|'):
        music_text += ' |'

    return music_text

def fix_abc_for_audio(abc_text):
    """Enhanced ABC fixer for audio conversion"""
    # First apply the basic structure fixes
    abc_text = fix_music_structure(abc_text)

    # Make sure we have a proper header
    if not re.search(r'X:', abc_text):
        abc_text = "X:1\nT:Generated Music\nM:4/4\nL:1/8\nQ:1/4=120\nK:C\n" + abc_text

    # Make sure we have a tempo marking to prevent extremely slow playback
    if not re.search(r'Q:', abc_text):
        # Insert tempo after the key
        abc_text = re.sub(r'(K:[^\n]+\n)', r'\1Q:1/4=120\n', abc_text)

    # Replace any problematic repeat sections that might cause loop issues
    abc_text = re.sub(r'\|\:\s*\|\:', '|', abc_text)

    # Ensure proper ending
    if not abc_text.endswith('|'):
        abc_text += ' |'

    return abc_text

def clean_abc_segment(segment):
    """Clean a small segment of ABC notation"""
    # Remove invalid characters
    cleaned = re.sub(r'[^ABCDEFGabcdefgz0-9\|\[\]\/\(\)\^\=\_\,\'\~\:\-\+ ]', '', segment)
    return cleaned

def convert_abc_to_audio(abc_text, filename_base, attempts=3):
    """Improved ABC to audio conversion with retry logic"""
    # Create output directory if needed
    output_dir = "audio_output"
    os.makedirs(output_dir, exist_ok=True)

    # Fully process the ABC text to ensure it's valid
    processed_abc = fix_abc_for_audio(abc_text)

    # Save processed ABC to file
    abc_file = f"{output_dir}/{filename_base}.abc"
    with open(abc_file, "w") as f:
        f.write(processed_abc)

    # Convert to WAV
    wav_file = f"{output_dir}/{filename_base}.wav"

    for attempt in range(attempts):
        try:
            print(f"Converting {filename_base} to WAV...")
            os.system(f"abc2midi {abc_file} -o {output_dir}/{filename_base}.mid")
            os.system(f"timidity {output_dir}/{filename_base}.mid -Ow -o {wav_file}")

            if os.path.exists(wav_file):
                print(f"Created WAV file: {wav_file}")
                return wav_file
            else:
                print(f"Attempt {attempt+1}: Failed to create WAV file")
        except Exception as e:
            print(f"Attempt {attempt+1}: Error processing {filename_base}: {e}")

    print(f"Failed to convert {filename_base} after {attempts} attempts")
    return None

def generate_long_song_with_gan(generator, latent_dim, idx2char, target_length=500, sequence_length=100):
    """Generate a longer song by chaining multiple GAN generations together"""
    # Initialize with a standard ABC notation header
    header = "X:1\nT:Generated Song\nM:4/4\nL:1/8\nK:C\n"

    # Generate the song in segments
    full_song = header
    current_length = len(header)

    with tqdm(total=target_length//sequence_length, desc="Generating segments") as pbar:
        while current_length < target_length:
            # Generate random noise as input to GAN
            noise = tf.random.normal([1, latent_dim])

            # Generate sequence with GAN
            generated_logits = generator(noise, training=False)

            # Convert logits to indices (most likely character at each position)
            generated_indices = tf.argmax(generated_logits[0], axis=-1).numpy()

            # Convert indices to characters
            generated_chars = [idx2char[idx] for idx in generated_indices]
            generated_segment = ''.join(generated_chars)

            # Clean the generated segment
            generated_segment = clean_abc_segment(generated_segment)

            # Add segment to the full song
            full_song += generated_segment
            current_length += len(generated_segment)

            # Update progress
            pbar.update(1)

            # Break if generated song is getting too long
            if current_length >= target_length * 2:
                break

    # Clean up the generated song for better playability
    full_song = fix_abc_for_audio(full_song)

    return full_song

if __name__ == "__main__":
    main()

## 1.8 Evaluation: Comparing Generated and Original Songs
### Metrics: Jaccard, Pitch Histogram, Repetition, n-gram, Contour,Coherence, BLEU Score, Sequence Similarity


In [ ]:
from music21 import converter, note
from collections import Counter
from difflib import SequenceMatcher
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np
import re

# --- Enhanced ABC fixing function ---
def fix_abc_notation_enhanced(abc_text):
    lines = abc_text.split('\n')
    fixed_lines = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Fix specific malformed headers
        if line.startswith('M::'):
            line = 'M:4/4'  # Default meter
        elif line.startswith('M:') and line == 'M:':
            line = 'M:4/4'
        elif line.startswith('L::'):
            line = 'L:1/8'  # Default note length
        elif line.startswith('L:') and line == 'L:':
            line = 'L:1/8'
        elif line.startswith('K::'):
            line = 'K:C'    # Default key
        elif line.startswith('K:') and line == 'K:':
            line = 'K:C'
        elif line.startswith('T::'):
            line = 'T:Generated Song'
        elif line.startswith('T:') and line == 'T:':
            line = 'T:Generated Song'
        elif line.startswith('X::'):
            line = 'X:1'
        elif line.startswith('X:') and line == 'X:':
            line = 'X:1'
        # Handle empty headers with just colon
        elif ':' in line and line.endswith(':'):
            header = line.split(':')[0]
            if header == 'M':
                line = 'M:4/4'
            elif header == 'L':
                line = 'L:1/8'
            elif header == 'K':
                line = 'K:C'
            elif header == 'T':
                line = 'T:Generated Song'
            elif header == 'X':
                line = 'X:1'

        fixed_lines.append(line)

    return '\n'.join(fixed_lines)

# --- Updated parsing function ---
def parse_abc_notes_fixed(abc_text, filename=""):
    print(f"\n=== PARSING ABC: {filename} ===")

    if not abc_text.strip():
        print("Empty ABC content")
        return []

    # Try direct parsing first
    try:
        print("Attempt 1: Direct music21 parsing...")
        parsed = converter.parse(abc_text, format='abc')
        notes = parsed.flatten().notes
        note_names = [n.nameWithOctave for n in notes if isinstance(n, note.Note)]
        print(f"Success! Extracted {len(note_names)} notes")
        return note_names
    except Exception as e:
        print(f"Failed: {e}")

    # Apply enhanced fixes and retry
    try:
        print("Attempt 2: Enhanced ABC fixing...")
        fixed_abc = fix_abc_notation_enhanced(abc_text)

        # Show what was fixed
        if fixed_abc != abc_text:
            print("Applied fixes:")
            original_lines = abc_text.split('\n')
            fixed_lines = fixed_abc.split('\n')
            for i, (orig, fixed) in enumerate(zip(original_lines, fixed_lines)):
                if orig != fixed:
                    print(f"  Line {i+1}: '{orig}' → '{fixed}'")

        parsed = converter.parse(fixed_abc, format='abc')
        notes = parsed.flatten().notes
        note_names = [n.nameWithOctave for n in notes if isinstance(n, note.Note)]
        print(f"Success after enhanced fixes! Extracted {len(note_names)} notes")
        return note_names
    except Exception as e:
        print(f"Enhanced fixing failed: {e}")

    # Fallback to manual extraction
    try:
        print("Attempt 3: Manual note extraction...")
        notes = extract_notes_manually(abc_text)
        if notes:
            print(f"✓ Manual extraction found {len(notes)} notes")
            return notes
    except Exception as e:
        print(f"Manual extraction failed: {e}")

    print("All parsing attempts failed")
    return []

# --- Manual note extraction (improved) ---
def extract_notes_manually(abc_text):
    lines = abc_text.split('\n')
    note_lines = []

    # Skip header lines and collect note content
    for line in lines:
        line = line.strip()
        # Skip empty lines and headers
        if not line or line.startswith(('X:', 'T:', 'M:', 'L:', 'K:', 'Q:', 'Z:', '%')):
            continue
        note_lines.append(line)

    if not note_lines:
        return []

    # Combine all note content and clean it
    note_content = ' '.join(note_lines)

    # Remove bar lines, repeat marks, and other symbols
    note_content = re.sub(r'[|\[\]!:]', ' ', note_content)

    # Extract notes with regex (handles various ABC notation)
    note_pattern = r'[A-Ga-g][#b]?[,\']*[0-9]*'
    matches = re.findall(note_pattern, note_content)

    # Convert to standard note names
    notes = []
    for match in matches:
        # Remove duration numbers
        clean_note = re.sub(r'[0-9]', '', match)
        if clean_note:  # Only process non-empty notes
            try:
                n = note.Note(clean_note)
                notes.append(n.nameWithOctave)
            except:
                continue

    return notes

# --- Keep all your original evaluation functions ---
def jaccard_overlap(a, b):
    set_a, set_b = set(a), set(b)
    return len(set_a & set_b) / len(set_a | set_b) if (set_a | set_b) else 0

def pitch_class_histogram(notes):
    h = {p: 0 for p in "CDEFGAB"}
    for n in notes:
        if n[0] in h:
            h[n[0]] += 1
    total = sum(h.values()) or 1
    return {k: v / total for k, v in h.items()}

def histogram_difference(h1, h2):
    return sum(abs(h1[k] - h2[k]) for k in h1)

def repetition_factor(notes):
    return sum(notes.count(n) for n in set(notes)) / len(notes) if notes else 0

def ngram_overlap(a, b, n=2):
    def extract_ngrams(seq, n):
        return Counter(tuple(seq[i:i+n]) for i in range(len(seq) - n + 1))
    a_ngrams = extract_ngrams(a, n)
    b_ngrams = extract_ngrams(b, n)
    shared = sum((a_ngrams & b_ngrams).values())
    total = sum((a_ngrams | b_ngrams).values())
    return shared / total if total else 0

def melodic_contour(notes):
    def to_midi(n):
        try: return note.Note(n).pitch.midi
        except: return None
    mids = [to_midi(n) for n in notes if n]
    mids = [m for m in mids if m is not None]
    contour = []
    for i in range(1, len(mids)):
        if mids[i] > mids[i-1]: contour.append('U')
        elif mids[i] < mids[i-1]: contour.append('D')
        else: contour.append('S')
    return contour

def contour_similarity(c1, c2):
    return SequenceMatcher(None, c1, c2).ratio()

def musical_coherence(notes):
    def to_midi(n):
        try: return note.Note(n).pitch.midi
        except: return None
    mids = [to_midi(n) for n in notes if n]
    mids = [m for m in mids if m is not None]
    if len(mids) < 2: return 1.0
    intervals = [mids[i+1] - mids[i] for i in range(len(mids) - 1)]
    std_dev = np.std(intervals)
    max_possible = 87
    return 1.0 - min(1.0, std_dev / max_possible)

def compute_bleu(reference, candidate):
    try:
        smoothing = SmoothingFunction().method1
        return sentence_bleu([reference], candidate, smoothing_function=smoothing)
    except Exception:
        return 0.0

def sequence_similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def evaluate_similarity(original_notes, generated_notes):
    if not original_notes or not generated_notes:
        print("One or both note sequences are empty. Check ABC content.")
        return

    print("\n MUSIC SIMILARITY EVALUATION")
    print("="*50)
    print(f" Original: {len(original_notes)} notes | Generated: {len(generated_notes)} notes")
    print(f"\n Similarity Metrics:")
    print(f"   Note Overlap (Jaccard):        {jaccard_overlap(original_notes, generated_notes):.3f}")

    h_orig = pitch_class_histogram(original_notes)
    h_gen = pitch_class_histogram(generated_notes)
    print(f"   Pitch Class Histogram Δ:       {histogram_difference(h_orig, h_gen):.3f}")
    print(f"   Repetition Factor (original):  {repetition_factor(original_notes):.3f}")
    print(f"   Repetition Factor (generated): {repetition_factor(generated_notes):.3f}")
    print(f"   Bigram Overlap:                {ngram_overlap(original_notes, generated_notes, 2):.3f}")
    print(f"   Trigram Overlap:               {ngram_overlap(original_notes, generated_notes, 3):.3f}")

    contour_orig = melodic_contour(original_notes)
    contour_gen = melodic_contour(generated_notes)
    print(f"   Melodic Contour Similarity:    {contour_similarity(contour_orig, contour_gen):.3f}")
    print(f"   Musical Coherence (original):  {musical_coherence(original_notes):.3f}")
    print(f"   Musical Coherence (generated): {musical_coherence(generated_notes):.3f}")
    print(f"   BLEU Score:                    {compute_bleu(original_notes, generated_notes):.3f}")
    print(f"   Sequence Similarity:           {sequence_similarity(original_notes, generated_notes):.3f}")

# --- Load ABC from file ---
def load_abc_from_file(filepath):
    try:
        with open(filepath, 'r') as f:
            return f.read()
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return ""

# --- Main execution ---
original_song = load_abc_from_file("/content/audio_output/1_original_song.abc")
reconstructed_song = load_abc_from_file("/content/audio_output/3_reconstructed_song.abc")

# Use the enhanced parsing function
original_notes = parse_abc_notes_fixed(original_song, "original")
reconstructed_notes = parse_abc_notes_fixed(reconstructed_song, "reconstructed")

evaluate_similarity(original_notes, reconstructed_notes)

#Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from music21 import converter, note
from collections import Counter
import re

# --- Enhanced ABC fixing function ---
def fix_abc_notation_enhanced(abc_text):
    lines = abc_text.split('\n')
    fixed_lines = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Fix specific malformed headers
        if line.startswith('M::'):
            line = 'M:4/4'
        elif line.startswith('M:') and line == 'M:':
            line = 'M:4/4'
        elif line.startswith('L::'):
            line = 'L:1/8'
        elif line.startswith('L:') and line == 'L:':
            line = 'L:1/8'
        elif line.startswith('K::'):
            line = 'K:C'
        elif line.startswith('K:') and line == 'K:':
            line = 'K:C'
        elif line.startswith('T::'):
            line = 'T:Generated Song'
        elif line.startswith('T:') and line == 'T:':
            line = 'T:Generated Song'
        elif line.startswith('X::'):
            line = 'X:1'
        elif line.startswith('X:') and line == 'X:':
            line = 'X:1'
        elif ':' in line and line.endswith(':'):
            header = line.split(':')[0]
            if header == 'M':
                line = 'M:4/4'
            elif header == 'L':
                line = 'L:1/8'
            elif header == 'K':
                line = 'K:C'
            elif header == 'T':
                line = 'T:Generated Song'
            elif header == 'X':
                line = 'X:1'

        fixed_lines.append(line)

    return '\n'.join(fixed_lines)

# --- Load ABC from file ---
def load_abc_from_file(filepath):
    try:
        with open(filepath, 'r') as f:
            return f.read()
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return ""

# --- Enhanced ABC parsing with error handling ---
def parse_abc_notes_fixed(abc_text, filename=""):
    print(f"\n=== PARSING ABC: {filename} ===")

    if not abc_text.strip():
        print(" Empty ABC content")
        return []

    # Try direct parsing first
    try:
        print("Attempt 1: Direct music21 parsing...")
        parsed = converter.parse(abc_text, format='abc')
        notes = parsed.flatten().notes
        note_names = [n.nameWithOctave for n in notes if isinstance(n, note.Note)]
        print(f"Success! Extracted {len(note_names)} notes")
        return note_names
    except Exception as e:
        print(f"Failed: {e}")

    # Apply enhanced fixes and retry
    try:
        print("Attempt 2: Enhanced ABC fixing...")
        fixed_abc = fix_abc_notation_enhanced(abc_text)

        # Show what was fixed
        if fixed_abc != abc_text:
            print("Applied fixes:")
            original_lines = abc_text.split('\n')
            fixed_lines = fixed_abc.split('\n')
            for i, (orig, fixed) in enumerate(zip(original_lines, fixed_lines)):
                if orig != fixed:
                    print(f"  Line {i+1}: '{orig}' → '{fixed}'")

        parsed = converter.parse(fixed_abc, format='abc')
        notes = parsed.flatten().notes
        note_names = [n.nameWithOctave for n in notes if isinstance(n, note.Note)]
        print(f" Success after enhanced fixes! Extracted {len(note_names)} notes")
        return note_names
    except Exception as e:
        print(f" Enhanced fixing failed: {e}")

    print(" All parsing attempts failed")
    return []

# --- Align sequences for fair comparison ---
def align_sequences(seq1, seq2):
    """Align sequences to same length for meaningful comparison"""
    min_length = min(len(seq1), len(seq2))
    return seq1[:min_length], seq2[:min_length]

# --- Create unified vocabulary ---
def create_unified_vocabulary(seq1, seq2):
    """Create unified vocabulary from both sequences"""
    all_notes = set(seq1 + seq2)
    vocab = sorted(list(all_notes))
    note_to_id = {note: i for i, note in enumerate(vocab)}
    return vocab, note_to_id

# --- Enhanced Confusion Matrix Plot ---
def plot_confusion_matrix(y_true_notes, y_pred_notes, title="Confusion Matrix"):
    """Enhanced confusion matrix that handles different vocabularies and sequence lengths"""

    if not y_true_notes or not y_pred_notes:
        print(" Cannot create confusion matrix: empty sequences")
        return

    print(f"\n Creating confusion matrix...")
    print(f"   Original: {len(y_true_notes)} notes")
    print(f"   Generated: {len(y_pred_notes)} notes")

    # Align sequences to same length
    y_true_aligned, y_pred_aligned = align_sequences(y_true_notes, y_pred_notes)
    print(f"   Aligned length: {len(y_true_aligned)}")

    if len(y_true_aligned) == 0:
        print(" No notes to compare after alignment")
        return

    # Create unified vocabulary
    vocab, note_to_id = create_unified_vocabulary(y_true_aligned, y_pred_aligned)

    # Convert to IDs
    y_true_ids = [note_to_id[note] for note in y_true_aligned]
    y_pred_ids = [note_to_id[note] for note in y_pred_aligned]

    # Create confusion matrix
    cm = confusion_matrix(y_true_ids, y_pred_ids, labels=range(len(vocab)))

    # Calculate accuracy
    accuracy = sum(1 for t, p in zip(y_true_ids, y_pred_ids) if t == p) / len(y_true_ids)

    # Create the plot
    plt.figure(figsize=(max(8, len(vocab) * 0.6), max(6, len(vocab) * 0.5)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=vocab, yticklabels=vocab)
    plt.xlabel("Predicted Notes")
    plt.ylabel("True Notes")
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# --- Example Usage ---
if __name__ == "__main__":
    print(" ABC Confusion Matrix Analysis")
    print("="*50)

    # Load the files
    original_song = load_abc_from_file("/content/audio_output/1_original_song.abc")
    reconstructed_song = load_abc_from_file("/content/audio_output/3_reconstructed_song.abc")

    # Parse using the correct function name
    original_notes = parse_abc_notes_fixed(original_song, "original")
    reconstructed_notes = parse_abc_notes_fixed(reconstructed_song, "reconstructed")

    # Create confusion matrix
    if original_notes and reconstructed_notes:
        plot_confusion_matrix(original_notes, reconstructed_notes,
                             "Original vs Reconstructed Notes")
    else:
        print(" Could not create confusion matrix: parsing failed")

#Audio Similarity(DTW)

In [ ]:
import librosa
import numpy as np
from librosa.sequence import dtw

# --- Audio Similarity using DTW over MFCCs ---
def compute_audio_similarity_dtw(file1, file2, sr=22050, n_mfcc=20):
    try:
        # Load audio
        y1, _ = librosa.load(file1, sr=sr)
        y2, _ = librosa.load(file2, sr=sr)

        # Extract MFCCs (shape: n_mfcc × time)
        mfcc1 = librosa.feature.mfcc(y=y1, sr=sr, n_mfcc=n_mfcc)
        mfcc2 = librosa.feature.mfcc(y=y2, sr=sr, n_mfcc=n_mfcc)

        # Compute DTW (match MFCC features over time using cosine distance)
        D, wp = dtw(X=mfcc1, Y=mfcc2, metric='cosine')

        # Normalize distance by length of path
        dist = D[-1, -1] / len(wp)
        similarity = 1 / (1 + dist)  # Higher is better
        print(f"DTW Audio Similarity: {similarity:.2f}")
        return similarity

    except Exception as e:
        print(f"Error during DTW audio comparison: {e}")
        return 0.0

# --- Example Usage ---
if __name__ == "__main__":
    file1 = "/content/audio_output/1_original_song.wav"
    file2 = "/content/audio_output/3_reconstructed_song.wav"
    compute_audio_similarity_dtw(file1, file2)

#Music Spectogram

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

# Load the original audio file
original_audio_path = "/content/audio_output/1_original_song.wav"  # Or "original_song.mp3"
y_orig, sr_orig = librosa.load(original_audio_path, sr=None)

# Create and display spectrogram for original music
plt.figure(figsize=(10, 4))
S_orig = librosa.feature.melspectrogram(y=y_orig, sr=sr_orig)
librosa.display.specshow(librosa.power_to_db(S_orig, ref=np.max),
                         sr=sr_orig, x_axis='time', y_axis='mel')
plt.title("Original Music Spectrogram")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()

# Load generated audio file
generated_audio_path = "/content/audio_output/3_reconstructed_song.wav"  # or .wav
y_gen, sr_gen = librosa.load(generated_audio_path, sr=None)

# Create and display spectrogram for generated music
plt.figure(figsize=(10, 4))
S_gen = librosa.feature.melspectrogram(y=y_gen, sr=sr_gen)
librosa.display.specshow(librosa.power_to_db(S_gen, ref=np.max),
                         sr=sr_gen, x_axis='time', y_axis='mel')
plt.title("Generated Music Spectrogram")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()

## Generating Multiple songs

In [ ]:
import tensorflow as tf
import numpy as np
import os
import re
import random
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.display import Audio, display

def main():
    print(" Loading dataset and GAN model...")
    import mitdeeplearning as mdl
    songs = mdl.lab1.load_training_data()

    vocab = sorted(set("".join(songs)))
    char2idx = {u: i for i, u in enumerate(vocab)}
    idx2char = np.array(vocab)

    # Load GAN model
    try:
        latent_dim = 100
        sequence_length = 100
        vocab_size = len(vocab)

        # Load the generator model
        generator = build_generator()
        generator.load_weights("/content/music_gan_checkpoints/generator_49500.weights.h5")
        print(" GAN generator model loaded successfully.")

    except Exception as e:
        print(f" Failed to load GAN model: {e}")
        return

    # Pick original song (for reconstruction/comparison if desired)
    original_song = random.choice(songs)
    original_length = len(original_song)

    print(f"\n Selected original song ({original_length} characters):")
    print(original_song[:200] + "...")

    # Generate multiple new songs
    num_songs_to_generate = 8
    generated_songs_data = []

    for i in range(num_songs_to_generate):
        print(f"\n Creating GENERATED song {i+1}/{num_songs_to_generate} (new composition)...")
        generated_song_text = generate_long_song_with_gan(
            generator, latent_dim, idx2char, target_length=original_length, sequence_length=sequence_length
        )
        generated_song_text = fix_abc_for_audio(generated_song_text)
        generated_songs_data.append({
            "text": generated_song_text,
            "filename_base": f"generated_song_{i+1}"
        })
        with open(f"generated_song_{i+1}.abc", "w") as f:
            f.write(generated_song_text)

    # Convert and play generated songs
    print("\n Converting generated songs to audio...")
    generated_audios = []
    for song_data in generated_songs_data:
        audio_file = convert_abc_to_audio(song_data["text"], song_data["filename_base"])
        if audio_file:
            generated_audios.append({"filename": audio_file, "title": f"Generated Song {i+1}"})

    print("\n🎧 GENERATED SONGS:")
    for i, audio_info in enumerate(generated_audios):
        print(f"\n {i+1}. {audio_info['title']}:")
        display(Audio(filename=audio_info['filename']))

def build_generator():
    """Build the generator model with identical architecture to the training model"""
    latent_dim = 100
    sequence_length = 100
    vocab_size = 83

    noise_input = tf.keras.layers.Input(shape=(latent_dim,))

    # Initial dense and reshape
    x = tf.keras.layers.Dense(256 * sequence_length, activation='relu')(noise_input)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Reshape((sequence_length, 256))(x)

    # Convolutional blocks with residual connections
    skip1 = x  # First skip connection

    # First conv block
    x = tf.keras.layers.Conv1D(256, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    # Second conv block
    x = tf.keras.layers.Conv1D(256, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    # Add skip connection
    x = tf.keras.layers.add([x, skip1])

    # Third conv block
    skip2 = x  # Second skip connection
    x = tf.keras.layers.Conv1D(128, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    # Fourth conv block
    x = tf.keras.layers.Conv1D(128, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    # Add skip connection (with projection if needed)
    skip2_projection = tf.keras.layers.Conv1D(128, 1, padding='same')(skip2)
    x = tf.keras.layers.add([x, skip2_projection])

    # Output layer
    output = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(vocab_size, activation='softmax'))(x)

    return tf.keras.models.Model(noise_input, output)

# === Function to Reconstruct the Original Song with GAN ===
def reconstruct_original_song_with_gan(generator, original_song, char2idx, idx2char, latent_dim=100, seq_length=100):
    """Direct reconstruction that preserves original musical structure using GAN"""
    print(" Reconstructing the original song through GAN...")

    # Extract header to preserve it exactly
    header_match = re.search(r'(X:.*?K:[^\n]+\n)', original_song, re.DOTALL)
    if header_match:
        header = header_match.group(1)
        body = original_song[len(header):]
    else:
        header = "X:1\nT:Reconstructed\nM:4/4\nL:1/8\nK:C\n"
        body = original_song

    # Add tempo if not present in header
    if "Q:" not in header:
        header = re.sub(r'(K:[^\n]+\n)', r'\1Q:1/4=120\n', header)

    # Extract the musical structure from the original - bar lines and repeated sections
    structure_template = re.sub(r'[A-Ga-g,\'0-9\^\_\=]', 'x', body)  # Replace notes with x

    # Process the body in small chunks with focus on musical motifs
    original_notes = re.findall(r'[A-Ga-g][,\'0-9]*', body)
    original_bars = body.split('|')

    print(f" Original song has {len(original_notes)} notes across {len(original_bars)} bars")

    # Direct approach: Use the GAN to generate each bar
    reconstructed_bars = []

    # For each bar in the original
    for i, bar in enumerate(tqdm(original_bars, desc="Processing bars")):
        if not bar.strip():  # Skip empty bars
            continue

        # Find notes in this bar
        bar_notes = re.findall(r'[A-Ga-g][,\'0-9]*', bar)
        if not bar_notes:  # Skip bars without notes
            reconstructed_bars.append(bar)
            continue

        # Generate random noise as input to GAN
        noise = tf.random.normal([1, latent_dim])

        # Generate sequence with GAN
        generated_logits = generator(noise, training=False)

        # Convert logits to indices (most likely character at each position)
        generated_indices = tf.argmax(generated_logits[0], axis=-1).numpy()

        # Convert indices to characters
        generated_chars = [idx2char[idx] for idx in generated_indices]
        generated_text = ''.join(generated_chars)

        # Extract notes from generated text
        generated_notes = re.findall(r'[A-Ga-g][,\'0-9]*', generated_text)

        # If no notes generated, use some from the original
        if not generated_notes and bar_notes:
            generated_notes = bar_notes

        # Recreate bar structure with generated notes
        bar_template = re.sub(r'[A-Ga-g][,\'0-9]*', 'NOTE', bar)
        reconstructed_bar = bar_template

        # Replace NOTE placeholders with actual notes
        for j, note in enumerate(generated_notes):
            if 'NOTE' in reconstructed_bar:
                reconstructed_bar = reconstructed_bar.replace('NOTE', note, 1)
            else:
                reconstructed_bar += note + " "

        # Ensure bar ends with |
        if not reconstructed_bar.endswith('|'):
            reconstructed_bar += ' |'

        reconstructed_bars.append(reconstructed_bar)

    # Combine bars
    reconstructed_body = ''.join(reconstructed_bars)

    # Check note count
    original_note_count = len(re.findall(r'[A-Ga-g]', body))
    reconstructed_note_count = len(re.findall(r'[A-Ga-g]', reconstructed_body))

    # If not enough notes, supplement with original material
    if reconstructed_note_count < original_note_count * 0.7:
        print(f" Still insufficient notes ({reconstructed_note_count} vs {original_note_count})")
        print(" Adding original motifs to enhance reconstruction")

        # Extract musical phrases from original
        phrases = []
        for bar in original_bars:
            if len(bar.strip()) > 3:  # Non-empty bar
                phrases.append(bar)

        # Add some original phrases
        sample_count = max(1, (original_note_count - reconstructed_note_count) // 20)
        selected_phrases = random.sample(phrases, min(sample_count, len(phrases)))

        for phrase in selected_phrases:
            reconstructed_body += phrase

    # Final processing
    reconstructed_body = fix_music_structure(reconstructed_body)

    # Complete song
    result = header + reconstructed_body

    # Final note count
    final_note_count = len(re.findall(r'[A-Ga-g]', result))

    print(f"Reconstructed {len(result)} characters (original was {len(original_song)})")
    print(f"Original notes: {original_note_count}, Reconstructed notes: {final_note_count})")

    return result

def fix_music_structure(music_text):
    """Fix common ABC notation structural issues"""
    # Ensure bar lines are properly spaced
    music_text = re.sub(r'\|\s*\|', '|', music_text)

    # Remove invalid characters often appearing in generated music
    music_text = re.sub(r'[^ABCDEFGabcdefgz0-9\|\[\]\/\(\)\^\=\_\,\'\~\:\-\+ ]', '', music_text)

    # Ensure proper bar line spacing
    music_text = re.sub(r'([A-Ga-g][0-9]?[,\']*)(?!\||\s)', r'\1 ', music_text)

    # Fix trailing bars
    if not music_text.endswith('|'):
        music_text += ' |'

    return music_text

def fix_abc_for_audio(abc_text):
    """Enhanced ABC fixer for audio conversion"""
    # First apply the basic structure fixes
    abc_text = fix_music_structure(abc_text)

    # Make sure we have a proper header
    if not re.search(r'X:', abc_text):
        abc_text = "X:1\nT:Generated Music\nM:4/4\nL:1/8\nQ:1/4=120\nK:C\n" + abc_text

    # Make sure we have a tempo marking to prevent extremely slow playback
    if not re.search(r'Q:', abc_text):
        # Insert tempo after the key
        abc_text = re.sub(r'(K:[^\n]+\n)', r'\1Q:1/4=120\n', abc_text)

    # Replace any problematic repeat sections that might cause loop issues
    abc_text = re.sub(r'\|\:\s*\|\:', '|', abc_text)

    # Ensure proper ending
    if not abc_text.endswith('|'):
        abc_text += ' |'

    return abc_text

def clean_abc_segment(segment):
    """Clean a small segment of ABC notation"""
    # Remove invalid characters
    cleaned = re.sub(r'[^ABCDEFGabcdefgz0-9\|\[\]\/\(\)\^\=\_\,\'\~\:\-\+ ]', '', segment)
    return cleaned

def convert_abc_to_audio(abc_text, filename_base, attempts=3):
    """Improved ABC to audio conversion with retry logic"""
    # Create output directory if needed
    output_dir = "audio_output"
    os.makedirs(output_dir, exist_ok=True)

    # Fully process the ABC text to ensure it's valid
    processed_abc = fix_abc_for_audio(abc_text)

    # Save processed ABC to file
    abc_file = f"{output_dir}/{filename_base}.abc"
    with open(abc_file, "w") as f:
        f.write(processed_abc)

    # Convert to WAV
    wav_file = f"{output_dir}/{filename_base}.wav"

    for attempt in range(attempts):
        try:
            print(f"Converting {filename_base} to WAV...")
            os.system(f"abc2midi {abc_file} -o {output_dir}/{filename_base}.mid")
            os.system(f"timidity {output_dir}/{filename_base}.mid -Ow -o {wav_file}")

            if os.path.exists(wav_file):
                print(f"Created WAV file: {wav_file}")
                return wav_file
            else:
                print(f"Attempt {attempt+1}: Failed to create WAV file")
        except Exception as e:
            print(f"Attempt {attempt+1}: Error processing {filename_base}: {e}")

    print(f"Failed to convert {filename_base} after {attempts} attempts")
    return None

def generate_long_song_with_gan(generator, latent_dim, idx2char, target_length=500, sequence_length=100):
    """Generate a longer song by chaining multiple GAN generations together"""
    # Initialize with a standard ABC notation header
    header = "X:1\nT:Generated Song\nM:4/4\nL:1/8\nK:C\n"

    # Generate the song in segments
    full_song = header
    current_length = len(header)

    with tqdm(total=target_length//sequence_length, desc="Generating segments") as pbar:
        while current_length < target_length:
            # Generate random noise as input to GAN
            noise = tf.random.normal([1, latent_dim])

            # Generate sequence with GAN
            generated_logits = generator(noise, training=False)

            # Convert logits to indices (most likely character at each position)
            generated_indices = tf.argmax(generated_logits[0], axis=-1).numpy()

            # Convert indices to characters
            generated_chars = [idx2char[idx] for idx in generated_indices]
            generated_segment = ''.join(generated_chars)

            # Clean the generated segment
            generated_segment = clean_abc_segment(generated_segment)

            # Add segment to the full song
            full_song += generated_segment
            current_length += len(generated_segment)

            # Update progress
            pbar.update(1)

            # Break if generated song is getting too long
            if current_length >= target_length * 2:
                break

    # Clean up the generated song for better playability
    full_song = fix_abc_for_audio(full_song)

    return full_song

if __name__ == "__main__":
    main()

##Calculating Average length

In [ ]:
import numpy as np

song_lengths = [len(song) for song in songs]
avg_song_length = np.mean(song_lengths)
min_song_length = np.min(song_lengths)
max_song_length = np.max(song_lengths)

print(f"Average song length: {avg_song_length:.2f} characters")
print(f"Minimum song length: {min_song_length} characters")
print(f"Maximum song length: {max_song_length} characters")